## Organize Data and Clean Where Necessary

### Goals
- import contribution data
- add header file
- import committee and candidate meta data
- convert data types
- export as parquet files

In [16]:
# imports

import pandas as pd
import numpy as np

In [28]:
cand_summ_dtypes = {
    "Link_Image": "string",
    "Cand_Name": "string",
    "Cand_Id": "string",
    "Cand_Office": "string",
    "Cand_Office_St": "string",
    "Cand_Office_Dist": "string",
    "Cand_Party_Affiliation": "string",
    "Cand_Incumbent_Challenger_Open_Seat": "string",
    "Coverage_Start_Date": "string",
    "Coverage_End_Date": "string",
    "Cand_Street_1": "string",
    "Cand_Street_2": "string",
    "Cand_City": "string",
    "Cand_State": "string",
    "Cand_Zip": "string",

    # Financial columns
    "Total_Receipt": "float64",
    "Total_Disbursement": "float64",
    "Cash_On_Hand_COP": "float64",
    "Cash_On_Hand_BOP": "float64",
    "Debt_Owed_By_Committee": "float64",
    "Debt_Owe_To_Committee": "float64",
    "Individual_Itemized_Contribution": "float64",
    "Individual_Unitemized_Contribution": "float64",
    "Individual_Contribution": "float64",
    "Other_Committee_Contribution": "float64",
    "Party_Committee_Contribution": "float64",
    "Cand_Contribution": "float64",
    "Total_Contribution": "float64",
    "Transfer_From_Other_Auth_Committee": "float64",
    "Cand_Loan": "float64",
    "Other_Loan": "float64",
    "Total_Loan": "float64",
    "Offsets_To_Operating_Expenditure": "float64",
    "Offsets_To_Fundraising": "float64",
    "Offsets_To_Leagal_Accounting": "float64",
    "Other_Receipts": "float64",
    "Operating_Expenditure": "float64",
    "Exempt_Legal_Accounting_Disbursement": "float64",
    "Fundraising_Disbursement": "float64",
    "Transfer_To_Other_Auth_Committee": "float64",
    "Cand_Loan_Repayment": "float64",
    "Other_Loan_Repayment": "float64",
    "Total_Loan_Repayment": "float64",
    "Individual_Refund": "float64",
    "Party_Committee_Refund": "float64",
    "Other_Committee_Refund": "float64",
    "Total_Contribution_Refund": "float64",
    "Other_Disbursements": "float64",
    "Net_Contribution": "float64",
    "Net_Operating_Expenditure": "float64"
}


comm_summ_dtypes = {
    "Link_Image": "string",
    "CMTE_ID": "string",
    "CMTE_NM": "string",
    "CMTE_TP": "string",
    "CMTE_DSGN": "string",
    "CMTE_FILING_FREQ": "string",
    "CMTE_ST1": "string",
    "CMTE_ST2": "string",
    "CMTE_CITY": "string",
    "CMTE_ST": "string",
    "CMTE_ZIP": "string",
    "TRES_NM": "string",
    "CAND_ID": "string",
    "FEC_ELECTION_YR": "Int64",
    "CVG_START_DT": "string",
    "CVG_END_DT": "string",
    "ORG_TP": "string",
    
    # all currency/financial columns as float
    "INDV_CONTB": "float64",
    "PTY_CMTE_CONTB": "float64",
    "OTH_CMTE_CONTB": "float64",
    "TTL_CONTB": "float64",
    "TRANF_FROM_OTHER_AUTH_CMTE": "float64",
    "OFFSETS_TO_OP_EXP": "float64",
    "OTHER_RECEIPTS": "float64",
    "TTL_RECEIPTS": "float64",
    "TRANF_TO_OTHER_AUTH_CMTE": "float64",
    "OTH_LOAN_REPYMTS": "float64",
    "INDV_REF": "float64",
    "POL_PTY_CMTE_REF": "float64",
    "TTL_CONTB_REF": "float64",
    "OTHER_DISB": "float64",
    "TTL_DISB": "float64",
    "NET_CONTB": "float64",
    "NET_OP_EXP": "float64",
    "COH_BOP": "float64",
    "COH_COP": "float64",
    "DEBTS_OWED_BY_CMTE": "float64",
    "DEBTS_OWED_TO_CMTE": "float64",
    "INDV_ITEM_CONTB": "float64",
    "INDV_UNITEM_CONTB": "float64",
    "OTH_LOANS": "float64",
    "TRANF_FROM_NONFED_ACCT": "float64",
    "TRANF_FROM_NONFED_LEVIN": "float64",
    "TTL_NONFED_TRANF": "float64",
    "LOAN_REPYMTS_RECEIVED": "float64",
    "OFFSETS_TO_FNDRSG": "float64",
    "OFFSETS_TO_LEGAL_ACCTG": "float64",
    "FED_CAND_CONTB_REF": "float64",
    "TTL_FED_RECEIPTS": "float64",
    "SHARED_FED_OP_EXP": "float64",
    "SHARED_NONFED_OP_EXP": "float64",
    "OTH_FED_OPE_EXP": "float64",
    "TTL_OP_EXP": "float64",
    "FED_CAND_CMTE_CONTB": "float64",
    "INDT_EXP": "float64",
    "COORD_EXP_BY_PTY_CMTE": "float64",
    "LOANS_MADE": "float64",
    "SHARED_FED_ACTVY_FED_SHR": "float64",
    "SHARED_FED_ACTVY_NONFED": "float64",
    "NON_ALLOC_FED_ELECT_ACTVY": "float64",
    "TTL_FED_ELECT_ACTVY": "float64",
    "TTL_FED_DISB": "float64",
    "CAND_CNTB": "float64",
    "CAND_LOAN": "float64",
    "TTL_LOANS": "float64",
    "OP_EXP": "float64",
    "CAND_LOAN_REPYMNT": "float64",
    "TTL_LOAN_REPYMTS": "float64",
    "OTH_CMTE_REF": "float64",
    "TTL_OFFSETS_TO_OP_EXP": "float64",
    "EXEMPT_LEGAL_ACCTG_DISB": "float64",
    "FNDRSG_DISB": "float64",
    "ITEM_REF_REB_RET": "float64",
    "SUBTTL_REF_REB_RET": "float64",
    "UNITEM_REF_REB_RET": "float64",
    "ITEM_OTHER_REF_REB_RET": "float64",
    "UNITEM_OTHER_REF_REB_RET": "float64",
    "SUBTTL_OTHER_REF_REB_RETB": "float64",
    "ITEM_OTHER_INCOME": "float64",
    "UNITEM_OTHER_INCOME": "float64",
    "EXP_PRIOR_YRS_SUBJECT_LIM": "float64",
    "EXP_SUBJECT_LIMITS": "float64",
    "FED_FUNDS": "float64",
    "ITEM_CONVN_EXP_DISB": "float64",
    "ITEM_OTHER_DISB": "float64",
    "SUBTTL_CONVN_EXP_DISB": "float64",
    "TTL_EXP_SUBJECT_LIMITS": "float64",
    "UNITEM_CONVN_EXP_DISB": "float64",
    "UNITEM_OTHER_DISB": "float64",
    "TTL_COMMUNICATION_COSTS": "float64",
    "COH_BOY": "float64",
    "COH_COY": "float64"
}


In [29]:
## csv to df converstion

cand_summ_22 = pd.read_csv('./raw_data/cand_summ_22.csv', dtype=comm_summ_dtypes, low_memory=False)
cand_summ_24 = pd.read_csv('./raw_data/cand_summ_24.csv', dtype=comm_summ_dtypes, low_memory=False)
comm_summ_22 = pd.read_csv('./raw_data/comm_summ_22.csv', dtype=comm_summ_dtypes, low_memory=False)
comm_summ_24 = pd.read_csv('./raw_data/comm_summ_24.csv', dtype=comm_summ_dtypes, low_memory=False)


In [30]:
comm_summ_22

,Link_Image,CMTE_ID,CMTE_NM,CMTE_TP,CMTE_DSGN,CMTE_FILING_FREQ,CMTE_ST1,CMTE_ST2,CMTE_CITY,CMTE_ST,...,ITEM_CONVN_EXP_DISB,ITEM_OTHER_DISB,SUBTTL_CONVN_EXP_DISB,TTL_EXP_SUBJECT_LIMITS,UNITEM_CONVN_EXP_DISB,UNITEM_OTHER_DISB,TTL_COMMUNICATION_COST,COH_BOY,COH_COY,ORG_TP
0,https://www.fec.gov/data/committee/C00810655/?...,C00810655,LOUISIANA HOME BUILDERS ASSOCIATION POLITICAL ...,N,U,Q,5015 RIVER ROAD,<NA>,HARAHAN,LA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T
1,https://www.fec.gov/data/committee/C00805309/?...,C00805309,SAVE GLENDORA SCHOOLS,N,U,Q,100 BARRANCA STREET,#742,WEST COVINA,CA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
2,https://www.fec.gov/data/committee/C00726828/?...,C00726828,HUDSON VICTORY FUND,N,J,Q,824 S. MILLEDGE AVE,SUITE 101,ATHENS,GA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
3,https://www.fec.gov/data/committee/C00727529/?...,C00727529,MRVAN FOR CONGRESS,H,P,Q,PO BOX 55,<NA>,CROWN POINT,IN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
4,https://www.fec.gov/data/committee/C00727123/?...,C00727123,CHRIS SMITH VICTORY FUND,N,J,Q,146 PROSPECT AVE,<NA>,TRENTON,NJ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13971,https://www.fec.gov/data/committee/C00464149/?...,C00464149,MO BROOKS FOR SENATE,S,P,Q,7610 FOXFIRE DR.,<NA>,HUNTSVILLE,AL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
13972,https://www.fec.gov/data/committee/C00464602/?...,C00464602,VICKY HARTZLER FOR SENATE,S,P,Q,PO BOX 531,<NA>,HARRISONVILLE,MO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
13973,https://www.fec.gov/data/committee/C00464602/?...,C00464602,VICKY HARTZLER FOR SENATE,S,P,Q,PO BOX 531,<NA>,HARRISONVILLE,MO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
13974,https://www.fec.gov/data/committee/C00464149/?...,C00464149,MO BROOKS FOR SENATE,S,P,Q,7610 FOXFIRE DR.,<NA>,HUNTSVILLE,AL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>


In [22]:
#bulk data files

cc_ie_22 = pd.read_csv('./raw_data/cc_ie_22.txt', sep='|', header=None, dtype=str)
cc_ie_24 = pd.read_csv('./raw_data/cc_ie_24.txt', sep='|', header=None, dtype=str)

cc_ie_header = pd.read_csv('./raw_data/cc_ie_header.csv')

cc_ie_22.columns = cc_ie_header.columns
cc_ie_24.columns = cc_ie_header.columns

dtype_map = {
    "CMTE_ID": "string",
    "AMNDT_IND": "string",
    "RPT_TP": "string",
    "TRANSACTION_PGI": "string",
    "IMAGE_NUM": "string",
    "TRANSACTION_TP": "string",
    "ENTITY_TP": "string",
    "NAME": "string",
    "CITY": "string",
    "STATE": "string",
    "ZIP_CODE": "string",
    "EMPLOYER": "string",
    "OCCUPATION": "string",
    "TRANSACTION_AMT": "float64",
    "OTHER_ID": "string",
    "CAND_ID": "string",
    "TRAN_ID": "string",
    "FILE_NUM": "Int64",  # pandas nullable int
    "MEMO_CD": "string",
    "MEMO_TEXT": "string",
    "SUB_ID": "Int64"
}

for col, dtype in dtype_map.items():
    if col in cc_ie_22.columns:
        cc_ie_22[col] = cc_ie_22[col].astype(dtype)

for col, dtype in dtype_map.items():
    if col in cc_ie_24.columns:
        cc_ie_22[col] = cc_ie_22[col].astype(dtype)

cc_ie_22["TRANSACTION_DT"] = pd.to_datetime(
    cc_ie_22["TRANSACTION_DT"], format="%m%d%Y", errors="coerce"
)

In [21]:
cc_ie_22['TRANSACTION_DT']

0         12242020
1         12242020
2         11172020
3         10272020
4         11172020
            ...   
751514    10252022
751515    11082022
751516    12012022
751517    12032022
751518    12032023
Name: TRANSACTION_DT, Length: 751519, dtype: object

### Save as Parquet

In [31]:
cand_summ_22.to_parquet('./datasets/cand_summ_22.parquet')
cand_summ_24.to_parquet('./datasets/cand_summ_24.parquet')
comm_summ_22.to_parquet('./datasets/comm_summ_22.parquet')
comm_summ_24.to_parquet('./datasets/comm_summ_24.parquet')
cc_ie_22.to_parquet('./datasets/cc_ie_22.parquet')
cc_ie_24.to_parquet('./datasets/cc_ie_24.parquet')